# Predição de Risco de Acidentes

Este notebook tem como objetivo criar um modelo preditivo para o risco de acidentes (`baixo`, `medio`, `alto`), com foco na classe de alto risco.

## Passos:
1. Carregamento e Preparação dos Dados
2. Separação de Dados de Validação (20%)
3. Pipelines de Pré-processamento
4. Tunning de Hiperparâmetros com Optuna (Logistic Regression vs LightGBM)
5. Avaliação dos Modelos (Separada)

In [198]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder, label_binarize, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score, make_scorer, roc_curve, auc, precision_recall_curve
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings('ignore')

## 1. Carregamento e Preparação dos Dados

In [199]:
# Carregar dados com Polars
df_pl = pl.read_parquet("data/anuario_prf.parquet")

# Converter para Pandas
df = df_pl.to_pandas()

# Visualizar primeiras linhas
df.head()

,uf,br,km,causa_acidente,tipo_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,veiculos,latitude,longitude,risco,data,mes,dia_semana_num
0,PR,116,33,Ingestão de álcool pelo condutor,Tombamento,Pleno dia,Decrescente,Nublado,Dupla,Curva,Não,3,2,-25.114403,-48.846755,alto,2022-01-01,1,6
1,MS,163,393,Condutor deixou de manter distância do veículo...,Colisão traseira,Amanhecer,Decrescente,Céu Claro,Simples,Aclive_Declive,Não,3,3,-21.228445,-54.456296,medio,2022-01-01,1,6
2,RJ,101,457,Reação tardia ou ineficiente do condutor,Colisão frontal,Pleno dia,Decrescente,Chuva,Simples,Curva,Sim,2,2,-23.031498,-44.177153,alto,2022-01-01,1,6
3,MG,40,508,Acumulo de água sobre o pavimento,Saída de leito carroçável,Pleno dia,Decrescente,Chuva,Dupla,Reta,Sim,3,1,-19.760612,-44.134754,baixo,2022-01-01,1,6
4,PB,116,8,Mal súbito do condutor,Colisão com objeto,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,3,2,-6.964668,-38.727608,baixo,2022-01-01,1,6


In [200]:
# Verificar tipos e nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258737 entries, 0 to 258736
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   uf                      258737 non-null  category      
 1   br                      258737 non-null  category      
 2   km                      258737 non-null  int64         
 3   causa_acidente          258737 non-null  category      
 4   tipo_acidente           258737 non-null  category      
 5   fase_dia                258737 non-null  category      
 6   sentido_via             258737 non-null  category      
 7   condicao_metereologica  258737 non-null  category      
 8   tipo_pista              258737 non-null  category      
 9   tracado_via             258737 non-null  category      
 10  uso_solo                258737 non-null  category      
 11  pessoas                 258737 non-null  int64         
 12  veiculos                258737

In [201]:
df.drop(columns=['causa_acidente','veiculos', 'pessoas', 'tipo_acidente'], inplace=True)

In [140]:
df

,uf,br,km,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,latitude,longitude,risco,data,mes,dia_semana_num
0,PR,116,33,Pleno dia,Decrescente,Nublado,Dupla,Curva,Não,-25.114403,-48.846755,alto,2022-01-01,1,6
1,MS,163,393,Amanhecer,Decrescente,Céu Claro,Simples,Aclive_Declive,Não,-21.228445,-54.456296,baixo,2022-01-01,1,6
2,RJ,101,457,Pleno dia,Decrescente,Chuva,Simples,Curva,Sim,-23.031498,-44.177153,medio,2022-01-01,1,6
3,MG,40,508,Pleno dia,Decrescente,Chuva,Dupla,Reta,Sim,-19.760612,-44.134754,baixo,2022-01-01,1,6
4,PB,116,8,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,-6.964668,-38.727608,baixo,2022-01-01,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258732,SC,282,302,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,-27.519538,-50.909615,medio,2025-09-27,9,6
258733,SP,381,82,Pleno dia,Crescente,Céu Claro,Múltipla,Reta,Sim,-23.481235,-46.562748,baixo,2025-08-30,8,6
258734,RJ,493,20,Pleno dia,Crescente,Sol,Simples,Curva,Não,-22.660571,-43.046321,medio,2025-08-25,8,1
258735,SC,101,321,Plena Noite,Crescente,Céu Claro,Dupla,Reta,Sim,-28.428350,-48.914460,medio,2025-09-28,9,7


In [175]:
df['dia_semana_num']

0         6
1         6
2         6
3         6
4         6
         ..
258732    6
258733    6
258734    1
258735    7
258736    6
Name: dia_semana_num, Length: 258737, dtype: int8

In [202]:
def criar_features_transito(df):
    df = df.copy()
    
    df['lat_cluster'] = df['latitude'].round(3).astype(str) + '_' + df['longitude'].round(3).astype(str)
    
    df['trecho_br'] = df['br'].astype(str) + '_' + (df['km'] // 10).astype(str)
    
    df['hora'] = df['data'].dt.hour
    
    df['mes_sin'] = np.sin(2 * np.pi * df['mes']/12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes']/12)
    
    df['sem_sin'] = np.sin(2 * np.pi * df['dia_semana_num']/7)
    df['sem_cos'] = np.cos(2 * np.pi * df['dia_semana_num']/7)
    
    df['is_weekend'] = df['dia_semana_num'].isin([5, 6, 7]).astype(int) # Ajuste conforme sua numeração (0-6 ou 1-7)
    
    df['pista_sentido'] = df['tipo_pista'].astype(str) + '_' + df['sentido_via'].astype(str)
    
    df['clima_tracado'] = df['condicao_metereologica'].astype(str) + '_' + df['tracado_via'].astype(str)
    
    return df

## 2. Separação de Dados de Validação

In [203]:
df_tratada = criar_features_transito(df)
X = df_tratada.drop(columns=['risco'])
y = df_tratada['risco']


In [204]:
df_tratada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258737 entries, 0 to 258736
Data columns (total 25 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   uf                      258737 non-null  category      
 1   br                      258737 non-null  category      
 2   km                      258737 non-null  int64         
 3   fase_dia                258737 non-null  category      
 4   sentido_via             258737 non-null  category      
 5   condicao_metereologica  258737 non-null  category      
 6   tipo_pista              258737 non-null  category      
 7   tracado_via             258737 non-null  category      
 8   uso_solo                258737 non-null  category      
 9   latitude                258737 non-null  float64       
 10  longitude               258737 non-null  float64       
 11  risco                   258737 non-null  category      
 12  data                    258737

In [205]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42
)

# Segundo: Dividir Temp em 50% Validação e 50% Teste (15% do total cada)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Salvar conjuntos
val_df = pd.concat([X_val, y_val], axis=1)
pl.from_pandas(val_df).write_parquet("data/validação.parquet")
test_df = pd.concat([X_test, y_test], axis=1)
pl.from_pandas(test_df).write_parquet("data/teste.parquet")

print(f"Treino shape: {X_train.shape}")
print(f"Validação shape: {X_val.shape}")
print(f"Teste shape: {X_test.shape}")


Treino shape: (155242, 24)
Validação shape: (51747, 24)
Teste shape: (51748, 24)


In [206]:
y_train_s1 = y_train.apply(lambda x: 0 if x == 'baixo' else 1)
y_val_s1 = y_val.apply(lambda x: 0 if x == 'baixo' else 1)

# Estágio 2: Médio (0) vs Alto (1) - Apenas para dados que NÃO são Baixo
# Filtrar dados de treino onde y original != 0
mask_train_s2 = y_train != 'baixo'
X_train_s2 = X_train[mask_train_s2]
y_train_s2 = y_train[mask_train_s2].map({'medio': 0, 'alto': 1}) # Mapear Médio->0, Alto->1

# Filtrar dados de validação (para uso no Optuna)
mask_val_s2 = y_val != 'baixo'
X_val_s2 = X_val[mask_val_s2]
y_val_s2 = y_val[mask_val_s2].map({'medio': 0, 'alto': 1})

print("Distribuição Estágio 1 (Treino):", y_train_s1.value_counts(normalize=True).to_dict())
print("Distribuição Estágio 2 (Treino):", y_train_s2.value_counts(normalize=True).to_dict())


Distribuição Estágio 1 (Treino): {1: 0.836004431790366, 0: 0.16399556820963399}
Distribuição Estágio 2 (Treino): {0.0: 0.6612345222409715, 1.0: 0.33876547775902854}


In [207]:
#Identificar colunas numéricas e categóricas
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['number']).columns.tolist()
numerical_cols.remove('hora')
categorical_cols.append('hora')

print("Numéricas:", numerical_cols)
print("Categóricas:", categorical_cols)

# Preprocessamento para Regressão Logística
numeric_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', TargetEncoder())
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_lr, numerical_cols),
        ('cat', categorical_transformer_lr, categorical_cols)
    ])

# Preprocessamento para LightGBM
numeric_transformer_lgbm = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer_lgbm = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('target_enc', TargetEncoder())
])

preprocessor_lgbm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_lgbm, numerical_cols),
        ('cat', categorical_transformer_lgbm, categorical_cols)
    ])

Numéricas: ['km', 'latitude', 'longitude', 'mes', 'dia_semana_num', 'mes_sin', 'mes_cos', 'sem_sin', 'sem_cos', 'is_weekend']
Categóricas: ['uf', 'br', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'lat_cluster', 'trecho_br', 'pista_sentido', 'clima_tracado', 'hora']


In [193]:
def objective_lr_s1(trial):
    # Estágio 1: Baixo vs Resto
    C = trial.suggest_loguniform('C', 1e-3, 100)
    solver = trial.suggest_categorical('solver', ['lbfgs'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    
    model = LogisticRegression(C=C, solver=solver, class_weight=class_weight, max_iter=1000, random_state=42)
    clf = Pipeline(steps=[('preprocessor', preprocessor_lr), ('classifier', model)])
    
    # F1 Macro para classificação binária
    score = cross_val_score(clf, X_train, y_train_s1, cv=5, scoring='accuracy', n_jobs=-1).mean()
    return score

def objective_lr_s2(trial):
    # Estágio 2: Médio vs Alto
    C = trial.suggest_loguniform('C', 1e-3, 100)
    solver = trial.suggest_categorical('solver', ['lbfgs'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    
    model = LogisticRegression(C=C, solver=solver, class_weight=class_weight, max_iter=1000, random_state=42)
    clf = Pipeline(steps=[('preprocessor', preprocessor_lr), ('classifier', model)])
    
    # F1 Macro para classificação binária
    score = cross_val_score(clf, X_train_s2, y_train_s2, cv=5, scoring='f1_macro', n_jobs=-1).mean()
    return score


In [196]:
def objective_lgbm_s1(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'n_jobs': 1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced'])
    }
    
    model = lgb.LGBMClassifier(**params)
    clf = Pipeline(steps=[('preprocessor', preprocessor_lgbm), ('classifier', model)])
    
    score = cross_val_score(clf, X_train, y_train_s1, cv=5, scoring='f1', n_jobs=-1).mean()
    return score

def objective_lgbm_s2(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'n_jobs': 1,
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced'])
    }
    
    model = lgb.LGBMClassifier(**params)
    clf = Pipeline(steps=[('preprocessor', preprocessor_lgbm), ('classifier', model)])
    
    score = cross_val_score(clf, X_train_s2, y_train_s2, cv=5, scoring='f1', n_jobs=-1).mean()
    return score


In [165]:
# Otimização LR
print("Otimizando LR Estágio 1...")
study_lr_s1 = optuna.create_study(direction='maximize')
study_lr_s1.optimize(objective_lr_s1, n_trials=40)

print("Otimizando LR Estágio 2...")
study_lr_s2 = optuna.create_study(direction='maximize')
study_lr_s2.optimize(objective_lr_s2, n_trials=40)



[I 2025-11-24 21:36:36,926] A new study created in memory with name: no-name-6c1054de-56e8-4716-b2fe-50276659422a


Otimizando LR Estágio 1...


[I 2025-11-24 21:36:45,224] Trial 0 finished with value: 0.6181445876678591 and parameters: {'C': 13.607873269611627, 'solver': 'lbfgs', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.6181445876678591.
[W 2025-11-24 21:36:47,340] Trial 1 failed with parameters: {'C': 0.0031636061902586064, 'solver': 'lbfgs', 'class_weight': 'balanced'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/aderson/miniconda3/envs/dengue/lib/python3.12/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_190709/3626814939.py", line 11, in objective_lr_s1
    score = cross_val_score(clf, X_train, y_train_s1, cv=5, scoring='accuracy', n_jobs=-1).mean()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aderson/miniconda3/envs/dengue/lib/python3.12/site-packages/sklearn/utils/_param_validation

KeyboardInterrupt: 

In [208]:
# Otimização LGBM
print("Otimizando LGBM Estágio 1...")
study_lgbm_s1 = optuna.create_study(direction='maximize')
study_lgbm_s1.optimize(objective_lgbm_s1, n_trials=20)

print("Otimizando LGBM Estágio 2...")
study_lgbm_s2 = optuna.create_study(direction='maximize')
study_lgbm_s2.optimize(objective_lgbm_s2, n_trials=20)


[I 2025-11-24 21:53:51,169] A new study created in memory with name: no-name-9a19dc52-f3de-444c-a088-dfe455365095


Otimizando LGBM Estágio 1...


[I 2025-11-24 21:54:01,184] Trial 0 finished with value: 0.6507624303160222 and parameters: {'learning_rate': 0.00561320530218321, 'num_leaves': 140, 'max_depth': 3, 'n_estimators': 90, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.6507624303160222.
[I 2025-11-24 21:54:19,203] Trial 1 finished with value: 0.7165337235674898 and parameters: {'learning_rate': 0.010689236051417276, 'num_leaves': 212, 'max_depth': 6, 'n_estimators': 321, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.7165337235674898.
[I 2025-11-24 21:54:44,565] Trial 2 finished with value: 0.7000005143420271 and parameters: {'learning_rate': 0.0011050020583537584, 'num_leaves': 150, 'max_depth': 9, 'n_estimators': 382, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.7165337235674898.
[I 2025-11-24 21:54:59,705] Trial 3 finished with value: 0.710056219909234 and parameters: {'learning_rate': 0.01280774192064632, 'num_leaves': 30, 'max_depth': 9, 'n_estimators': 473, 'class_weight': 'bala

Otimizando LGBM Estágio 2...


[I 2025-11-24 22:00:06,262] Trial 0 finished with value: 0.5064782454719688 and parameters: {'learning_rate': 0.04387957937772512, 'num_leaves': 232, 'max_depth': 4, 'n_estimators': 299, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.5064782454719688.
[I 2025-11-24 22:00:22,050] Trial 1 finished with value: 0.4999361877867214 and parameters: {'learning_rate': 0.0019237532281892184, 'num_leaves': 222, 'max_depth': 8, 'n_estimators': 290, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.5064782454719688.
[I 2025-11-24 22:00:42,775] Trial 2 finished with value: 0.5000539342471548 and parameters: {'learning_rate': 0.01804331441118457, 'num_leaves': 195, 'max_depth': 8, 'n_estimators': 419, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.5064782454719688.
[I 2025-11-24 22:00:51,340] Trial 3 finished with value: 0.5054939034505358 and parameters: {'learning_rate': 0.021728892052637717, 'num_leaves': 66, 'max_depth': 11, 'n_estimators': 166, 'class_weight': 'b

In [117]:
def hierarchical_predict(model_s1, model_s2, X):
    # Previsão Estágio 1: 0 (Baixo) vs 1 (Resto)
    pred_s1 = model_s1.predict(X)
    
    # Inicializar array final com 0 (Baixo)
    final_preds = np.zeros_like(pred_s1)
    
    # Identificar índices onde a previsão foi 'Resto' (1)
    mask_resto = (pred_s1 == 1)
    
    if np.any(mask_resto):
        # Previsão Estágio 2 apenas para esses casos: 0 (Médio) vs 1 (Alto)
        X_resto = X[mask_resto]
        pred_s2 = model_s2.predict(X_resto)
        
        # Mapear de volta: 0->1 (Médio), 1->2 (Alto)
        final_preds[mask_resto] = pred_s2 + 1
        
    return final_preds


In [118]:
def evaluate_hierarchical(model_s1, model_s2, X_test, y_test, name):
    print(f"--- Avaliação Hierárquica: {name} ---")
    
    # Gera as previsões (que retornam ints 0, 1, 2)
    y_pred = hierarchical_predict(model_s1, model_s2, X_test)
    
    # --- CORREÇÃO: Tratamento do y_test ---
    # Se y_test não for numérico, precisamos converter
    y_test_proc = y_test.copy()
    
    # Verifica se é do tipo objeto/string e converte
    if y_test_proc.dtype == 'object':
        # Mapeamento manual (garante que 'baixo' seja 0, etc.)
        # Ajuste as chaves aqui se seus labels originais forem diferentes (ex: "Baixo", "LOW", etc)
        mapping = {'baixo': 0, 'medio': 1, 'alto': 2}
        
        # Se o y_test for apenas strings de números ("0", "1"), o map retornará NaN, 
        # então usamos um try/except ou verificação direta.
        try:
            # Tenta mapear nomes para números
            if set(mapping.keys()).intersection(set(y_test_proc.unique())):
                y_test_proc = y_test_proc.map(mapping)
            else:
                # Se não tiver as chaves, tenta converter direto (caso sejam strings "0", "1")
                y_test_proc = y_test_proc.astype(int)
        except ValueError:
            print("Erro: Não foi possível converter y_test para numérico automaticamente.")
    # ---------------------------------------

    print("\nRelatório de Classificação (Teste):")
    # Agora passamos y_test_proc que é garantidamente numérico
    print(classification_report(y_test_proc, y_pred, target_names=['baixo', 'medio', 'alto']))
    
    print("\nMatriz de Confusão:")
    cm = confusion_matrix(y_test_proc, y_pred)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['baixo', 'medio', 'alto'], 
                yticklabels=['baixo', 'medio', 'alto'])
    plt.title(f'Matriz de Confusão - {name}')
    plt.ylabel('Real')
    plt.xlabel('Predito')
    plt.show()

In [ ]:


# Treinar e Avaliar LR Hierárquico
print("Treinando Modelos Finais LR...")
lr_s1 = LogisticRegression(max_iter=1000, random_state=42, **study_lr_s1.best_params)
pipe_lr_s1 = Pipeline(steps=[('preprocessor', preprocessor_lr), ('classifier', lr_s1)])
pipe_lr_s1.fit(X_train, y_train_s1)

lr_s2 = LogisticRegression(max_iter=1000, random_state=42, **study_lr_s2.best_params)
pipe_lr_s2 = Pipeline(steps=[('preprocessor', preprocessor_lr), ('classifier', lr_s2)])
pipe_lr_s2.fit(X_train_s2, y_train_s2)

evaluate_hierarchical(pipe_lr_s1, pipe_lr_s2, X_test, y_test, "Logistic Regression Hierárquica")


Treinando Modelos Finais LR...
--- Avaliação Hierárquica: Logistic Regression Hierárquica ---


ValueError: X has 180 features, but LogisticRegression is expecting 181 features as input.

In [115]:

# Treinar e Avaliar LGBM Hierárquico
print("Treinando Modelos Finais LGBM...")
lgbm_s1 = lgb.LGBMClassifier(objective='binary', metric='binary_logloss', verbosity=-1, boosting_type='gbdt', random_state=42, **study_lgbm_s1.best_params)
pipe_lgbm_s1 = Pipeline(steps=[('preprocessor', preprocessor_lgbm), ('classifier', lgbm_s1)])
pipe_lgbm_s1.fit(X_train, y_train_s1)

lgbm_s2 = lgb.LGBMClassifier(objective='binary', metric='binary_logloss', verbosity=-1, boosting_type='gbdt', random_state=42, **study_lgbm_s2.best_params)
pipe_lgbm_s2 = Pipeline(steps=[('preprocessor', preprocessor_lgbm), ('classifier', lgbm_s2)])
pipe_lgbm_s2.fit(X_train_s2, y_train_s2)



Treinando Modelos Finais LGBM...


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
